# Backtest Markowitz — predicciones ARIMA (agrupación semanal por suma)

Igual que el backtest anterior pero usando el retorno semanal calculado como suma de retornos logarítmicos diarios. La covarianza y los retornos históricos se escalan en consecuencia. Se comparan cuatro configuraciones de riesgo distintas: diversificada, intermedia, agresiva y muy concentrada.

In [1]:
# pip install PyPortfolioOpt cvxpy osqp gspread google-auth

In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
from datetime import datetime
from pypfopt import risk_models, EfficientFrontier

import gspread
from google.oauth2.service_account import Credentials

warnings.filterwarnings('ignore')

## Configuración

In [ ]:
DATA_DIR             = '../Datos_csv'
SERVICE_ACCOUNT_FILE = '../3.Modelo_ML/Modelos_semanales/credenciales_google.json'
SHEET_URL            = 'https://docs.google.com/spreadsheets/d/1SMWy1XzpUiMBZ0caEiQ9li6a2ZRU0kvjH5WQT8N4Rx4/edit?gid=0#gid=0'

SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]

# Split temporal
FECHA_TRAIN_FIN = '2024-W29'
FECHA_TEST_INI  = '2024-W30'
FECHA_TEST_FIN  = '2026-W05'
CAPITAL_INICIAL = 100.0
MIN_WEEKS       = 52

# ── Las 4 configuraciones finales ─────────────────────────────────────────────
# RISK_AVERSION escalado para suma semanal (~5x respecto a media)
# Ejemplo: RA=0.10 con suma ≡ RA≈0.5 con media
CONFIGURACIONES = [
    {'nombre': 'diversificada',   'MAX_WEIGHT': 0.10, 'RISK_AVERSION': 0.10, 'hoja_detalle': 'detalle_diversificada'},
    {'nombre': 'intermedia',      'MAX_WEIGHT': 0.20, 'RISK_AVERSION': 0.40, 'hoja_detalle': 'detalle_intermedia'},
    {'nombre': 'agresiva',        'MAX_WEIGHT': 0.30, 'RISK_AVERSION': 0.10, 'hoja_detalle': 'detalle_agresiva'},
    {'nombre': 'muy_concentrada', 'MAX_WEIGHT': 0.50, 'RISK_AVERSION': 0.10, 'hoja_detalle': 'detalle_muy_concentrada'},
]

HOJA_RESUMEN = 'resumen_configs'

## Carga de datos

In [4]:
# Precios ajustados
adj_close = pd.read_csv(
    os.path.join(DATA_DIR, 'adj_close.csv'),
    index_col=0, parse_dates=True
)

# ETFs válidos (pasan filtro de estacionalidad)
with open(os.path.join(DATA_DIR, 'tickers_estacionalidad.json'), 'r', encoding='utf-8') as f:
    est_data = json.load(f)
tickers_filtrados = est_data['tickers_filtrados']

# Retornos logarítmicos diarios solo para ETFs válidos
ret_log_diff = np.log(adj_close[tickers_filtrados]).diff().iloc[1:]

# Clave semana ISO (YYYY-Wnn)
iso_cal  = ret_log_diff.index.isocalendar()
week_key = (
    iso_cal.year.astype(str) + '-W' + iso_cal.week.astype(str).str.zfill(2)
).values

# ── Retorno semanal = SUMA de retornos logarítmicos diarios de la semana ──
ret_weekly_df = ret_log_diff.groupby(week_key).sum()
ret_weekly_df.index.name = 'week_key'

print(f'ETFs validos: {len(tickers_filtrados)}')
print(f'Semanas totales: {len(ret_weekly_df)} | {ret_weekly_df.index[0]} -> {ret_weekly_df.index[-1]}')
ret_weekly_df.head(3)

ETFs validos: 58
Semanas totales: 266 | 2021-W01 -> 2026-W06


,AGG,BND,DBC,DIA,DVY,EEM,EFA,EWG,EWJ,EWQ,...,XLB,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
week_key,,,,,,,,,,,,,,,,,,,,,
2021-W01,-0.008166,-0.009125,0.041533,0.028436,0.048207,0.050803,0.032068,0.023357,0.037197,0.024671,...,0.064931,0.087231,0.061353,0.036363,0.022115,0.002694,0.007883,0.019604,0.038556,0.057113
2021-W02,0.001451,0.000917,0.003929,-0.009020,0.014576,-0.006602,-0.017718,-0.033357,-0.008260,-0.029440,...,-0.015410,0.031606,0.000647,-0.008742,-0.025875,-0.019166,0.018889,0.010533,-0.003584,-0.017873
2021-W03,0.000000,0.000343,-0.013820,0.005987,-0.008760,0.026867,0.010030,0.022054,0.007538,0.000299,...,-0.012824,-0.016034,-0.019583,-0.003609,0.041581,-0.008569,0.013122,-0.002384,0.005286,0.025604


In [5]:
# ── CSV de predicciones ARIMA para los 58 ETFs (sin filtrar) ─────────────
arima_path = os.path.join(DATA_DIR, 'Prueba_predicciones_arima_semanal.csv')
df_arima = pd.read_csv(arima_path, encoding='utf-8-sig')

# Pivote: semana x ETF -> predicción ARIMA
pred_arima_pivot = df_arima.pivot(
    index='Fecha_Semana', columns='Target', values='Prediccion'
)
pred_arima_pivot.index.name = 'week_key'

print(f'Predicciones ARIMA: {pred_arima_pivot.shape[1]} ETFs, {pred_arima_pivot.shape[0]} semanas')
print(f'Rango: {pred_arima_pivot.index[0]} -> {pred_arima_pivot.index[-1]}')

Predicciones ARIMA: 58 ETFs, 81 semanas
Rango: 2024-W30 -> 2026-W06


## Optimizador Markowitz

$$\max_w \; w^\top \mu - \frac{\lambda}{2} w^\top \Sigma w$$
Sujeto a: $\sum w_i = 1$, $0 \leq w_i \leq w_{\max}$

Covarianza estimada con **Ledoit-Wolf** (shrinkage)

In [6]:
def calcular_covarianza_lw(ret_hist: pd.DataFrame) -> pd.DataFrame:
    Sigma = risk_models.CovarianceShrinkage(
        ret_hist, returns_data=True, frequency=1
    ).ledoit_wolf()
    return risk_models.fix_nonpositive_semidefinite(Sigma)


# Detectar solver una sola vez
_solvers = set(cp.installed_solvers())
if 'OSQP' in _solvers:
    _SOLVER, _SOLVER_OPTS = 'OSQP', {'max_iter': 300000}
elif 'SCS' in _solvers:
    _SOLVER, _SOLVER_OPTS = 'SCS',  {'max_iters': 100000}
else:
    raise RuntimeError(f'Sin solver QP. Instalados: {sorted(_solvers)}')
print(f'Solver: {_SOLVER}')


def optimizar_markowitz(
    mu: np.ndarray,
    cov: pd.DataFrame,
    etfs: list,
    max_weight: float = 0.30,
    risk_aversion: float = 2.0
) -> dict:
    n = len(etfs)
    mu_series = pd.Series(mu, index=etfs)
    try:
        ef = EfficientFrontier(
            mu_series, cov,
            weight_bounds=(0, max_weight),
            solver=_SOLVER,
            solver_options=_SOLVER_OPTS,
        )
        ef.max_quadratic_utility(risk_aversion=2 * risk_aversion)
        return dict(ef.clean_weights())
    except Exception as e:
        print(f'  [FALLBACK equal-weight] {e}')
        return {etf: 1.0 / n for etf in etfs}


def retorno_cartera(pesos: dict, ret_real: pd.Series) -> float:
    ret_simple = np.exp(ret_real) - 1
    return sum(pesos.get(etf, 0.0) * ret_simple.get(etf, 0.0) for etf in pesos)

Solver: OSQP


## Backtest semanal

Para cada semana `t` del periodo de test:
1. **Covarianza**: todos los retornos disponibles hasta `t` (Ledoit-Wolf) — universo 58 ETFs.
2. **Rentabilidad esperada**: según la estrategia.
3. **Optimización**: cartera Markowitz.
4. **Inversión**: pesos aplicados a retornos reales de `t+1`.
5. **Actualización**: capital acumulado.

In [ ]:
semanas_test = [
    s for s in ret_weekly_df.index
    if FECHA_TEST_INI <= s <= FECHA_TEST_FIN
]
etfs_comunes = [e for e in tickers_filtrados if e in ret_weekly_df.columns]
print(f'Semanas de backtest: {len(semanas_test)-1} | {semanas_test[0]} -> {semanas_test[-1]}')
print(f'ETFs en cartera: {len(etfs_comunes)}')

# ── Precomputar cov/mu por semana (compartidos entre las 4 configs) ────────
print('\nPrecomputando covarianzas y medias...')
semanas_validas = []
cache_cov    = {}
cache_media  = {}
cache_ultimo = {}

for i, semana_decision in enumerate(semanas_test[:-1]):
    semana_inversion = semanas_test[i + 1]
    hist = ret_weekly_df.loc[
        ret_weekly_df.index <= semana_decision, etfs_comunes
    ].dropna(how='all')
    if len(hist) < MIN_WEEKS:
        continue
    if semana_inversion not in ret_weekly_df.index:
        continue
    try:
        cov = calcular_covarianza_lw(hist[etfs_comunes].dropna())
    except Exception as e:
        print(f'[{semana_decision}] Error covarianza: {e}')
        continue
    semanas_validas.append((i, semana_decision, semana_inversion))
    cache_cov[semana_decision]    = cov
    cache_media[semana_decision]  = hist[etfs_comunes].mean().values
    cache_ultimo[semana_decision] = hist[etfs_comunes].iloc[-1].values

print(f'Semanas válidas: {len(semanas_validas)}')

# ── Bucle: 4 configs × semanas ─────────────────────────────────────────────
nombres_bt = {
    'media_historica': 'Markowitz_media_historica',
    'ultimo_valor':    'Markowitz_ultimo_valor',
    'arima':           'Markowitz_ARIMA_sin_filtrar',
}

resultados_configs = {}

for cfg in CONFIGURACIONES:
    MAX_WEIGHT    = cfg['MAX_WEIGHT']
    RISK_AVERSION = cfg['RISK_AVERSION']
    nombre_cfg    = cfg['nombre']

    print(f"\n{'='*60}")
    print(f"  Config: {nombre_cfg.upper()} | MAX_WEIGHT={MAX_WEIGHT} | RISK_AVERSION={RISK_AVERSION}")
    print(f"{'='*60}")

    registros    = {k: [] for k in nombres_bt}
    capital      = {k: CAPITAL_INICIAL for k in nombres_bt}
    capital_pico = {k: CAPITAL_INICIAL for k in nombres_bt}

    for idx, (i, semana_decision, semana_inversion) in enumerate(semanas_validas):
        cov       = cache_cov[semana_decision]
        mu_media  = cache_media[semana_decision]
        mu_ultimo = cache_ultimo[semana_decision]
        ret_real  = ret_weekly_df.loc[semana_inversion, etfs_comunes]

        mu_arima = (
            pred_arima_pivot.loc[semana_inversion, etfs_comunes].values
            if semana_inversion in pred_arima_pivot.index
            else mu_media
        )

        estrategias = [
            ('media_historica', optimizar_markowitz(mu_media,  cov, etfs_comunes, MAX_WEIGHT, RISK_AVERSION), mu_media),
            ('ultimo_valor',    optimizar_markowitz(mu_ultimo, cov, etfs_comunes, MAX_WEIGHT, RISK_AVERSION), mu_ultimo),
            ('arima',           optimizar_markowitz(mu_arima,  cov, etfs_comunes, MAX_WEIGHT, RISK_AVERSION), mu_arima),
        ]

        for nombre, pesos, mu_used in estrategias:
            ret_cart  = retorno_cartera(pesos, ret_real)
            cap_ini   = capital[nombre]
            cap_fin   = cap_ini * (1.0 + ret_cart)
            capital[nombre]      = cap_fin
            capital_pico[nombre] = max(capital_pico[nombre], cap_fin)
            drawdown  = (cap_fin - capital_pico[nombre]) / capital_pico[nombre]
            rent_acum = (cap_fin - CAPITAL_INICIAL) / CAPITAL_INICIAL

            row = {
                'Config':                nombre_cfg,
                'MAX_WEIGHT':            MAX_WEIGHT,
                'RISK_AVERSION':         RISK_AVERSION,
                'FechaDecision':         semana_decision,
                'FechaInversion':        semana_inversion,
                'Estrategia':            nombre,
                'CapitalInicio':         round(cap_ini, 4),
                'RetornoCarteraReal':    round(ret_cart, 6),
                'CapitalFinal':          round(cap_fin, 4),
                'RentabilidadAcumulada': round(rent_acum, 6),
                'Drawdown':              round(drawdown, 6),
                'SumaPesos':             round(sum(pesos.values()), 6),
                'Estado':                'DONE',
            }
            for etf in etfs_comunes:
                row[f'{etf}_peso']    = round(pesos.get(etf, 0.0), 6)
            for j, etf in enumerate(etfs_comunes):
                row[f'{etf}_rentexp'] = round(float(mu_used[j]), 6)
            for etf in etfs_comunes:
                val = ret_real.get(etf, np.nan)
                row[f'{etf}_real'] = round(float(val), 6) if not np.isnan(val) else None

            registros[nombre].append(row)

        if (idx + 1) % 20 == 0:
            print(f'  Semana {idx+1:3d}/{len(semanas_validas)}: {semana_decision} -> {semana_inversion}')

    # ── Métricas ────────────────────────────────────────────────────────────
    dfs = {k: pd.DataFrame(v) for k, v in registros.items()}
    ids_bt = {
        'media_historica': f'{nombre_cfg}_MH',
        'ultimo_valor':    f'{nombre_cfg}_UV',
        'arima':           f'{nombre_cfg}_AR',
    }

    resumen_rows = []
    for nombre, df_e in dfs.items():
        if df_e.empty:
            continue
        rets   = df_e['RetornoCarteraReal'].values
        cap_f  = df_e['CapitalFinal'].iloc[-1]
        vol    = np.std(rets, ddof=1) * np.sqrt(52)
        sharpe = (np.mean(rets) * 52) / (vol + 1e-10)
        caps   = df_e['CapitalFinal'].values
        peak, max_dd = caps[0], 0.0
        for c in caps:
            peak = max(peak, c)
            max_dd = min(max_dd, (c - peak) / peak)
        resumen_rows.append({
            'ID':                    ids_bt[nombre],
            'Config':                nombre_cfg,
            'MAX_WEIGHT':            MAX_WEIGHT,
            'RISK_AVERSION':         RISK_AVERSION,
            'Estrategia':            nombres_bt[nombre],
            'ETFs_universo':         len(etfs_comunes),
            'Agregacion_semanal':    'suma',
            'FechaTrainIni':         '2021-W01',
            'FechaTrainFin':         FECHA_TRAIN_FIN,
            'FechaTestIni':          FECHA_TEST_INI,
            'FechaTestFin':          df_e['FechaInversion'].iloc[-1],
            'CapitalInicial':        CAPITAL_INICIAL,
            'CapitalFinal':          round(cap_f, 4),
            'RentabilidadAcumulada': f'{(cap_f-CAPITAL_INICIAL)/CAPITAL_INICIAL*100:.2f}%',
            'Volatilidad':           f'{vol*100:.2f}%',
            'Sharpe':                round(sharpe, 4),
            'MaxDrawdown':           f'{max_dd*100:.2f}%',
            'SemanasPositivas':      int((rets > 0).sum()),
            'Estado':                'DONE',
        })

    df_resumen_cfg = pd.DataFrame(resumen_rows)
    print(f"\n  Resultados {nombre_cfg.upper()}:")
    print(df_resumen_cfg[['Estrategia', 'CapitalFinal', 'RentabilidadAcumulada',
                           'Volatilidad', 'Sharpe', 'MaxDrawdown']].to_string(index=False))

    resultados_configs[nombre_cfg] = {
        'cfg':        cfg,
        'dfs':        dfs,
        'ids_bt':     ids_bt,
        'df_resumen': df_resumen_cfg,
    }

print("\n\n=== TODAS LAS CONFIGURACIONES COMPLETADAS ===")

In [8]:
# Resumen rápido por config
for nombre_cfg, res in resultados_configs.items():
    cfg = res['cfg']
    print(f"\n{nombre_cfg.upper()} (MW={cfg['MAX_WEIGHT']}, RA={cfg['RISK_AVERSION']}):")
    for k, df_e in res['dfs'].items():
        if not df_e.empty:
            cap_f = df_e['CapitalFinal'].iloc[-1]
            print(f"  {k}: {len(df_e)} semanas | Capital final: {cap_f:.4f}")


BASE (MW=0.3, RA=0.4):
  media_historica: 79 semanas | Capital final: 126.9876
  ultimo_valor: 79 semanas | Capital final: 119.1349
  arima: 79 semanas | Capital final: 119.6290

CONSERVADOR (MW=0.15, RA=1.0):
  media_historica: 79 semanas | Capital final: 120.8498
  ultimo_valor: 79 semanas | Capital final: 115.0543
  arima: 79 semanas | Capital final: 118.2812

AGRESIVO (MW=0.5, RA=0.1):
  media_historica: 79 semanas | Capital final: 118.2983
  ultimo_valor: 79 semanas | Capital final: 117.3504
  arima: 79 semanas | Capital final: 111.3951


## Métricas de resumen

In [9]:
# Tabla consolidada de métricas de todas las configs
df_resumen_total = pd.concat(
    [res['df_resumen'] for res in resultados_configs.values()],
    ignore_index=True
)
display(df_resumen_total[['ID', 'Config', 'MAX_WEIGHT', 'RISK_AVERSION',
                           'Estrategia', 'CapitalFinal', 'RentabilidadAcumulada',
                           'Volatilidad', 'Sharpe', 'MaxDrawdown', 'SemanasPositivas']])

,ID,Config,MAX_WEIGHT,RISK_AVERSION,Estrategia,CapitalFinal,RentabilidadAcumulada,Volatilidad,Sharpe,MaxDrawdown,SemanasPositivas
0,base_MH,base,0.30,0.4,Markowitz_media_historica,126.9876,26.99%,20.71%,0.8643,-17.12%,48
1,base_UV,base,0.30,0.4,Markowitz_ultimo_valor,119.1349,19.13%,20.63%,0.6609,-21.82%,47
2,base_AR,base,0.30,0.4,Markowitz_ARIMA_sin_filtrar,119.6290,19.63%,15.38%,0.8442,-16.49%,45
3,conservador_MH,conservador,0.15,1.0,Markowitz_media_historica,120.8498,20.85%,15.78%,0.8702,-15.29%,48
4,conservador_UV,conservador,0.15,1.0,Markowitz_ultimo_valor,115.0543,15.05%,13.93%,0.7324,-13.59%,47
5,conservador_AR,conservador,0.15,1.0,Markowitz_ARIMA_sin_filtrar,118.2812,18.28%,12.81%,0.9273,-14.76%,45
6,agresivo_MH,agresivo,0.50,0.1,Markowitz_media_historica,118.2983,18.30%,27.20%,0.5421,-17.37%,45
7,agresivo_UV,agresivo,0.50,0.1,Markowitz_ultimo_valor,117.3504,17.35%,25.74%,0.5365,-24.87%,47
8,agresivo_AR,agresivo,0.50,0.1,Markowitz_ARIMA_sin_filtrar,111.3951,11.40%,18.94%,0.4703,-18.92%,48


## Evolución del capital

In [ ]:
colores = {
    'media_historica': '#4FC3F7',
    'ultimo_valor':    '#81C784',
    'arima':           '#EF739A',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes_flat = axes.flatten()

for ax, cfg in zip(axes_flat, CONFIGURACIONES):
    nombre_cfg = cfg['nombre']
    res = resultados_configs[nombre_cfg]
    for nombre, df_e in res['dfs'].items():
        if not df_e.empty:
            ax.plot(df_e['FechaInversion'], df_e['CapitalFinal'],
                    color=colores[nombre], linewidth=1.8)
    ax.axhline(CAPITAL_INICIAL, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(
        f"{nombre_cfg.upper()}\nMW={cfg['MAX_WEIGHT']} | RA={cfg['RISK_AVERSION']}",
        fontweight='bold', fontsize=10
    )
    ax.set_ylabel('Capital')
    handles = [
        plt.Line2D([0], [0], color=colores[k], linewidth=2,
                   label={'media_historica': 'Media histórica',
                           'ultimo_valor': 'Último valor',
                           'arima': 'ARIMA'}[k])
        for k in colores
    ]
    ax.legend(handles=handles, fontsize=7)
    ax.grid(alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=6)

plt.suptitle('Evolución del capital — 4 configs finales (58 ETFs, suma semanal)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## Escritura en Google Sheets

**Hoja resumen_backtest** (gid=0): una fila por estrategia con métricas globales.  
**Hoja detalle_semanal**: una fila por semana × estrategia con pesos, rentabilidades esperadas y retornos reales.

In [11]:
creds       = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
client      = gspread.authorize(creds)
spreadsheet = client.open_by_url(SHEET_URL)

hojas_disponibles = [ws.title for ws in spreadsheet.worksheets()]
print('Conectado a:', spreadsheet.title)
print('Hojas disponibles:')
for h in hojas_disponibles:
    print(f'  → "{h}"')

Conectado a: Copia Markowitz Sin filtrar
Hojas disponibles:
  → "resumen_configs"
  → "detalle_base"
  → "detalle_conservador"
  → "detalle_agresivo"


In [ ]:
# ── Hoja resumen_configs ──────────────────────────────────────────────────────
ws_resumen = spreadsheet.worksheet(HOJA_RESUMEN)
ws_resumen.clear()

df_resumen_total = pd.concat(
    [res['df_resumen'] for res in resultados_configs.values()],
    ignore_index=True
)

ws_resumen.append_row(list(df_resumen_total.columns), value_input_option='RAW')
for _, row in df_resumen_total.iterrows():
    ws_resumen.append_row(
        [v if isinstance(v, (int, float)) else str(v) for v in row.tolist()],
        value_input_option='RAW'
    )
print(f'Hoja "{HOJA_RESUMEN}": {len(df_resumen_total)} filas escritas (4 configs × 3 estrategias)')

In [13]:
# ── Hoja detalle por config ───────────────────────────────────────────────────
cols_base = [
    'ID_Backtest', 'Config', 'MAX_WEIGHT', 'RISK_AVERSION',
    'FechaDecision', 'FechaInversion', 'Estrategia',
    'CapitalInicio', 'RetornoCarteraReal', 'CapitalFinal',
    'RentabilidadAcumulada', 'Drawdown', 'SumaPesos', 'Estado',
]
cols_pesos   = [f'{e}_peso'    for e in etfs_comunes]
cols_rentexp = [f'{e}_rentexp' for e in etfs_comunes]
cols_real    = [f'{e}_real'    for e in etfs_comunes]
all_cols = cols_base + cols_pesos + cols_rentexp + cols_real

for nombre_cfg, res in resultados_configs.items():
    hoja_det = res['cfg']['hoja_detalle']
    ws_det   = spreadsheet.worksheet(hoja_det)
    ws_det.clear()
    ws_det.append_row(all_cols, value_input_option='RAW')

    rows_batch = []
    for nombre, df_e in res['dfs'].items():
        for _, row in df_e.iterrows():
            fila = [res['ids_bt'][nombre]]
            for col in cols_base[1:]:
                val = row.get(col, '')
                fila.append('' if (isinstance(val, float) and np.isnan(val)) else val)
            for col in cols_pesos + cols_rentexp + cols_real:
                val = row.get(col, '')
                fila.append('' if (val is None or (isinstance(val, float) and np.isnan(val))) else val)
            rows_batch.append(fila)

    batch_size = 500
    for i in range(0, len(rows_batch), batch_size):
        ws_det.append_rows(rows_batch[i:i+batch_size], value_input_option='RAW')

    print(f'Hoja "{hoja_det}": {len(rows_batch)} filas escritas')

Hoja "detalle_base": 237 filas escritas
Hoja "detalle_conservador": 237 filas escritas
Hoja "detalle_agresivo": 237 filas escritas


In [ ]:
print('=== RESUMEN COMPARATIVO FINAL — 4 CONFIGS (58 ETFs, suma semanal) ===')
print(f'{"Config":<18} {"Estrategia":<30} {"Capital":>8} {"Rent":>8} {"Sharpe":>7} {"MaxDD":>9}')
print('-' * 84)
for nombre_cfg, res in resultados_configs.items():
    for _, row in res['df_resumen'].iterrows():
        print(
            f"{nombre_cfg:<18} {row['Estrategia']:<30} "
            f"{row['CapitalFinal']:>8.2f} {row['RentabilidadAcumulada']:>8} "
            f"{row['Sharpe']:>7.4f} {row['MaxDrawdown']:>9}"
        )